# 天気図パターン分類 - すぐに使う

リポジトリに同梱されている学習済みモデル(`weights/model.pt`)を使って、
アップロードした天気図画像がどの気圧配置パターンに近いかを判定します。
学習は行わず、Google Driveのマウントも不要です。

上から順にセルを実行してください。

## 1. セットアップ

In [ ]:
REPO_URL = "https://github.com/awg-yk/weather-pattern-classification.git"
BRANCH = "claude/weather-chart-classification-4b6in1"
REPO_DIR = "/content/weather-pattern-classification"
WEIGHTS_PATH = f"{REPO_DIR}/weights/model.pt"  # リポジトリに同梱されているモデル

import subprocess, os

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!apt-get -qq install -y fonts-noto-cjk

assert os.path.exists(WEIGHTS_PATH), f"モデルの重みが見つかりません: {WEIGHTS_PATH}"
print("セットアップ完了。モデル:", WEIGHTS_PATH)

In [ ]:
import sys
sys.path.append(REPO_DIR)

from src.labels import LABEL_JA
from scripts.gradcam import explain_top_prediction

print("準備完了")

## 2. 画像をアップロードして推論

このセルを実行するたびに新しい画像をアップロードして分類できます。
一番確信度が高いラベルについては、モデルが画像のどこに注目したかを
ヒートマップ(Grad-CAM)で色付き表示します。残りの候補は確信度の数値のみ表示します。

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# apply_preprocess=True: 気象庁の生のPDF変換画像(枠・座標グリッド・日時スタンプ付き)を想定。
# 既に前処理済みの画像を使う場合はFalseにしてください。
display_image, top_overlay, ranked = explain_top_prediction(
    image_path=image_path,
    weights_path=WEIGHTS_PATH,
    apply_preprocess=True,
)

top_label, top_prob = ranked[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(display_image)
axes[0].set_title("入力画像(前処理後)")
axes[0].axis("off")

axes[1].imshow(top_overlay)
axes[1].set_title(f"予測: {LABEL_JA[top_label]} ({top_prob * 100:.1f}%)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("--- 全ラベルの確信度 ---")
for label, prob in ranked:
    print(f"{LABEL_JA[label]}: {prob * 100:.1f}%")